In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [7]:
import pandas as pd
import sqlite3
import json
from pathlib import Path
from tqdm import tqdm

In [8]:
DATA_DIR = Path("/kaggle/input/datasets/sherrynamembwa/patent-pipeline2003-2008")
DB_PATH = Path("/kaggle/working/patent_pipeline.db")
CHUNK_SIZE = 100000
print("Data dir exists:", DATA_DIR.exists())
print("Files:", list(DATA_DIR.glob("*.tsv"))[:5])

Data dir exists: True
Files: [PosixPath('/kaggle/input/datasets/sherrynamembwa/patent-pipeline2003-2008/g_inventor_disambiguated.tsv'), PosixPath('/kaggle/input/datasets/sherrynamembwa/patent-pipeline2003-2008/g_location_not_disambiguated.tsv'), PosixPath('/kaggle/input/datasets/sherrynamembwa/patent-pipeline2003-2008/g_assignee_not_disambiguated.tsv'), PosixPath('/kaggle/input/datasets/sherrynamembwa/patent-pipeline2003-2008/g_assignee_disambiguated.tsv'), PosixPath('/kaggle/input/datasets/sherrynamembwa/patent-pipeline2003-2008/g_persistent_assignee.tsv')]


In [9]:
def read_tsv_in_chunks(file_path, columns=None, chunksize=CHUNK_SIZE):
    with open(file_path, 'r') as f:
        chunk_iter = pd.read_csv(f, sep='\t', usecols=columns, chunksize=chunksize, low_memory=False)
        for chunk in chunk_iter:
            yield chunk
print("Helper defined")

Helper defined


In [10]:
import os
if DB_PATH.exists():
    DB_PATH.unlink()
conn = sqlite3.connect(DB_PATH)
conn.executescript("""
    CREATE TABLE patents (patent_id TEXT PRIMARY KEY, title TEXT, abstract TEXT, filing_date TEXT, year INTEGER);
    CREATE TABLE inventors (inventor_id TEXT PRIMARY KEY, name TEXT, country TEXT);
    CREATE TABLE companies (company_id TEXT PRIMARY KEY, name TEXT);
    CREATE TABLE patent_inventor (patent_id TEXT, inventor_id TEXT, PRIMARY KEY (patent_id, inventor_id));
    CREATE TABLE patent_company (patent_id TEXT, company_id TEXT, PRIMARY KEY (patent_id, company_id));
""")
conn.commit()
print("Schema created")

Schema created


In [11]:
def build_patents(conn):
    print("Building patents table...")
    patent_abstracts = {}
    abstract_path = DATA_DIR / "g_patent_abstract.tsv"
    if abstract_path.exists():
        for chunk in tqdm(read_tsv_in_chunks(abstract_path, columns=['patent_id', 'patent_abstract']), desc="Abstracts"):
            for _, row in chunk.iterrows():
                patent_abstracts[row['patent_id']] = row['patent_abstract']
    else:
        print("Abstract file not found")
        return
    patent_path = DATA_DIR / "g_patent.tsv"
    if not patent_path.exists():
        print("Patent file not found")
        return
    for chunk in tqdm(read_tsv_in_chunks(patent_path, columns=['patent_id', 'patent_title', 'patent_date']), desc="Patents"):
        chunk = chunk.rename(columns={'patent_title': 'title', 'patent_date': 'filing_date'})
        chunk['year'] = pd.to_datetime(chunk['filing_date'], errors='coerce').dt.year
        chunk['abstract'] = chunk['patent_id'].map(patent_abstracts).fillna("No abstract")
        chunk[['patent_id', 'title', 'abstract', 'filing_date', 'year']].to_sql('patents', conn, if_exists='append', index=False)
    conn.execute("CREATE INDEX IF NOT EXISTS idx_patents_year ON patents(year);")
    print(" Patents done")

build_patents(conn)

Building patents table...


Abstracts: 95it [06:40,  4.22s/it]
Patents: 95it [12:49,  8.10s/it]


 Patents done


In [12]:
def build_inventors(conn):
    print("Building inventors table...")
    
    # Load location map (location_id → country)
    loc_map = {}
    loc_path = DATA_DIR / "g_location_disambiguated.tsv"
    if loc_path.exists():
        for chunk in tqdm(read_tsv_in_chunks(loc_path, columns=['location_id', 'disambig_country']), desc="Locations"):
            for _, row in chunk.iterrows():
                loc_map[row['location_id']] = row['disambig_country']
        print(f"  Loaded {len(loc_map)} locations")
    else:
        print("  Warning: location file not found, countries will be 'Unknown'")
    
    # Process inventors
    inv_path = DATA_DIR / "g_inventor_disambiguated.tsv"
    if not inv_path.exists():
        print("  Error: g_inventor_disambiguated.tsv not found")
        return
    
    seen_ids = set()
    total_inserted = 0
    for chunk in tqdm(read_tsv_in_chunks(inv_path, columns=['inventor_id', 'disambig_inventor_name_first', 'disambig_inventor_name_last', 'location_id']), desc="Inventors"):
        # Create full name
        chunk['name'] = (chunk['disambig_inventor_name_first'].fillna('') + ' ' + chunk['disambig_inventor_name_last'].fillna('')).str.strip()
        chunk.loc[chunk['name'] == '', 'name'] = 'Unknown'
        # Add country from location map
        chunk['country'] = chunk['location_id'].map(loc_map).fillna('Unknown')
        # Keep only needed columns and deduplicate within chunk
        df_inv = chunk[['inventor_id', 'name', 'country']].drop_duplicates('inventor_id')
        # Avoid duplicates across chunks
        df_inv = df_inv[~df_inv['inventor_id'].isin(seen_ids)]
        seen_ids.update(df_inv['inventor_id'])
        if not df_inv.empty:
            df_inv.to_sql('inventors', conn, if_exists='append', index=False)
            total_inserted += len(df_inv)
    
    conn.execute("CREATE INDEX IF NOT EXISTS idx_inventors_id ON inventors(inventor_id);")
    print(f" Inventors done – inserted {total_inserted} unique inventors")

build_inventors(conn)

Building inventors table...


Locations: 2it [00:03,  1.80s/it]


  Loaded 100452 locations


Inventors: 241it [16:32,  4.12s/it]


 Inventors done – inserted 4294034 unique inventors


In [13]:
def build_companies(conn):
    print("Building companies table...")
    
    assignee_path = DATA_DIR / "g_assignee_not_disambiguated.tsv"
    if not assignee_path.exists():
        print("  Error: g_assignee_not_disambiguated.tsv not found")
        return
    
    seen_ids = set()
    total_inserted = 0
    for chunk in tqdm(read_tsv_in_chunks(assignee_path, columns=['assignee_id', 'raw_assignee_organization']), desc="Companies"):
        # Rename columns to match schema
        chunk = chunk.rename(columns={'assignee_id': 'company_id', 'raw_assignee_organization': 'name'})
        # Drop rows with missing company_id or empty name
        chunk = chunk.dropna(subset=['company_id'])
        chunk['name'] = chunk['name'].fillna('Unknown')
        # Deduplicate within chunk
        df_comp = chunk[['company_id', 'name']].drop_duplicates('company_id')
        # Avoid duplicates across chunks
        df_comp = df_comp[~df_comp['company_id'].isin(seen_ids)]
        seen_ids.update(df_comp['company_id'])
        if not df_comp.empty:
            df_comp.to_sql('companies', conn, if_exists='append', index=False)
            total_inserted += len(df_comp)
    
    conn.execute("CREATE INDEX IF NOT EXISTS idx_companies_id ON companies(company_id);")
    print(f" Companies done – inserted {total_inserted} unique companies")

build_companies(conn)

Building companies table...


Companies: 88it [01:00,  1.45it/s]


 Companies done – inserted 572495 unique companies


In [15]:
#conn.execute("DROP TABLE IF EXISTS patent_inventor")
#conn.execute("""
#    CREATE TABLE patent_inventor (
#        patent_id TEXT,
#        inventor_id TEXT,
#        PRIMARY KEY (patent_id, inventor_id)
#    )
#""")
#conn.commit()

In [16]:
def build_patent_inventor(conn):
    print("Building patent-inventor links...")
    rel_path = DATA_DIR / "g_inventor_disambiguated.tsv"
    if not rel_path.exists():
        print("  Error: g_inventor_disambiguated.tsv not found")
        return
    
    seen_pairs = set()
    total_inserted = 0
    for chunk in tqdm(read_tsv_in_chunks(rel_path, columns=['patent_id', 'inventor_id']), desc="Inventor links"):
        # Remove duplicates within chunk
        chunk = chunk.drop_duplicates()
        chunk = chunk.dropna(subset=['patent_id', 'inventor_id'])
        
        # Build a list of (patent_id, inventor_id) tuples for this chunk
        pairs = list(zip(chunk['patent_id'], chunk['inventor_id']))
        # Keep only pairs not seen in previous chunks
        mask = [pair not in seen_pairs for pair in pairs]
        chunk = chunk[mask]
        # Update the set of seen pairs
        seen_pairs.update(pairs[i] for i, keep in enumerate(mask) if keep)
        
        if not chunk.empty:
            chunk.to_sql('patent_inventor', conn, if_exists='append', index=False)
            total_inserted += len(chunk)
    
    conn.execute("CREATE INDEX IF NOT EXISTS idx_pi_patent ON patent_inventor(patent_id);")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_pi_inventor ON patent_inventor(inventor_id);")
    print(f" Patent-inventor done – inserted {total_inserted} links")

build_patent_inventor(conn)

Building patent-inventor links...


Inventor links: 241it [28:30,  7.10s/it]


 Patent-inventor done – inserted 24035239 links


In [17]:
def build_patent_company(conn):
    print("Building patent-company links...")
    rel_path = DATA_DIR / "g_assignee_not_disambiguated.tsv"
    if not rel_path.exists():
        print("  Error: g_assignee_not_disambiguated.tsv not found")
        return
    
    seen_pairs = set()
    total_inserted = 0
    for chunk in tqdm(read_tsv_in_chunks(rel_path, columns=['patent_id', 'assignee_id']), desc="Company links"):
        chunk = chunk.rename(columns={'assignee_id': 'company_id'})
        chunk = chunk.drop_duplicates()
        chunk = chunk.dropna(subset=['patent_id', 'company_id'])
        
        pairs = list(zip(chunk['patent_id'], chunk['company_id']))
        mask = [pair not in seen_pairs for pair in pairs]
        chunk = chunk[mask]
        seen_pairs.update(pairs[i] for i, keep in enumerate(mask) if keep)
        
        if not chunk.empty:
            chunk.to_sql('patent_company', conn, if_exists='append', index=False)
            total_inserted += len(chunk)
    
    conn.execute("CREATE INDEX IF NOT EXISTS idx_pc_patent ON patent_company(patent_id);")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_pc_company ON patent_company(company_id);")
    print(f" Patent-company done – inserted {total_inserted} links")

build_patent_company(conn)

Building patent-company links...


Company links: 88it [07:21,  5.02s/it]


 Patent-company done – inserted 8748737 links


In [18]:
for table in ['patents', 'inventors', 'companies', 'patent_inventor', 'patent_company']:
    count = pd.read_sql_query(f"SELECT COUNT(*) FROM {table}", conn).iloc[0][0]
    print(f"{table}: {count} rows")

/tmp/ipykernel_55/1404019894.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  count = pd.read_sql_query(f"SELECT COUNT(*) FROM {table}", conn).iloc[0][0]


patents: 9454161 rows
inventors: 4294034 rows
companies: 572495 rows
patent_inventor: 24035239 rows
patent_company: 8748737 rows


In [ ]:
def run_queries(conn):
    print("\n" + "="*60)
    print(" RUNNING REQUIRED SQL QUERIES")
    print("="*60)
    
    queries = {
        "Q1_Top_Inventors": """
            SELECT i.name, COUNT(pi.patent_id) as patent_count
            FROM inventors i
            JOIN patent_inventor pi ON i.inventor_id = pi.inventor_id
            GROUP BY i.inventor_id
            ORDER BY patent_count DESC
            LIMIT 10
        """,
        "Q2_Top_Companies": """
            SELECT c.name, COUNT(pc.patent_id) as patent_count
            FROM companies c
            JOIN patent_company pc ON c.company_id = pc.company_id
            GROUP BY c.company_id
            ORDER BY patent_count DESC
            LIMIT 10
        """,
        "Q3_Top_Countries": """
            SELECT i.country, COUNT(DISTINCT pi.patent_id) as patent_count
            FROM inventors i
            JOIN patent_inventor pi ON i.inventor_id = pi.inventor_id
            GROUP BY i.country
            ORDER BY patent_count DESC
            LIMIT 10
        """,
        "Q4_Trends_Over_Time": """
            SELECT year, COUNT(*) as patent_count
            FROM patents
            WHERE year IS NOT NULL
            GROUP BY year
            ORDER BY year
        """,
        "Q5_Join_Query": """
            SELECT p.patent_id, p.title, i.name as inventor_name, c.name as company_name
            FROM patents p
            LEFT JOIN patent_inventor pi ON p.patent_id = pi.patent_id
            LEFT JOIN inventors i ON pi.inventor_id = i.inventor_id
            LEFT JOIN patent_company pc ON p.patent_id = pc.patent_id
            LEFT JOIN companies c ON pc.company_id = c.company_id
            LIMIT 20
        """,
        "Q6_CTE_Query": """
            WITH yearly_inventor_counts AS (
                SELECT i.inventor_id, i.name, strftime('%Y', p.filing_date) as year, COUNT(*) as cnt
                FROM inventors i
                JOIN patent_inventor pi ON i.inventor_id = pi.inventor_id
                JOIN patents p ON pi.patent_id = p.patent_id
                GROUP BY i.inventor_id, year
            )
            SELECT name, year, cnt
            FROM yearly_inventor_counts
            WHERE cnt > 1
            ORDER BY year DESC, cnt DESC
            LIMIT 20
        """,
        "Q7_Ranking_Query": """
            SELECT i.name, COUNT(pi.patent_id) as patent_count,
                   RANK() OVER (ORDER BY COUNT(pi.patent_id) DESC) as rank
            FROM inventors i
            JOIN patent_inventor pi ON i.inventor_id = pi.inventor_id
            GROUP BY i.inventor_id
            ORDER BY rank
            LIMIT 20
        """
    }
    
    results = {}
    for name, sql in queries.items():
        print(f"\n--- {name} ---")
        df = pd.read_sql_query(sql, conn)
        print(df.to_string(index=False))
        results[name] = df
    return results

def generate_reports(conn, results):
    print("\n" + "="*60)
    print(" EXPORTING REPORTS")
    print("="*60)
    
    # CSV reports
    results["Q1_Top_Inventors"].to_csv("/kaggle/working/top_inventors.csv", index=False)
    results["Q2_Top_Companies"].to_csv("/kaggle/working/top_companies.csv", index=False)
    results["Q3_Top_Countries"].to_csv("/kaggle/working/country_trends.csv", index=False)
    
    # Clean CSVs
    clean_patents = pd.read_sql_query("SELECT * FROM patents", conn)
    clean_inventors = pd.read_sql_query("SELECT * FROM inventors", conn)
    clean_companies = pd.read_sql_query("SELECT * FROM companies", conn)
    clean_patents.to_csv("/kaggle/working/clean_patents.csv", index=False)
    clean_inventors.to_csv("/kaggle/working/clean_inventors.csv", index=False)
    clean_companies.to_csv("/kaggle/working/clean_companies.csv", index=False)
    
    # JSON report
    total_patents = pd.read_sql_query("SELECT COUNT(*) as cnt FROM patents", conn).iloc[0]["cnt"]
    top_inventors = results["Q1_Top_Inventors"].head(5).to_dict(orient="records")
    top_companies = results["Q2_Top_Companies"].head(5).to_dict(orient="records")
    top_countries = results["Q3_Top_Countries"].head(5).to_dict(orient="records")
    
    json_report = {
    "total_patents": int(total_patents),
    "top_inventors": [{"name": r["name"], "patents": int(r["patent_count"])} for r in top_inventors],
    "top_companies": [{"name": r["name"], "patents": int(r["patent_count"])} for r in top_companies],
    "top_countries": [{"country": r["country"], "patent_count": int(r["patent_count"])} for r in top_countries]
}
    
    with open("/kaggle/working/report.json", "w") as f:
        json.dump(json_report, f, indent=2)
    
    print(" All reports saved to /kaggle/working/")

# Run the queries and reports
results = run_queries(conn)
generate_reports(conn, results)
conn.close()
print("\n Pipeline complete!")


 RUNNING REQUIRED SQL QUERIES

--- Q1_Top_Inventors ---
                    name  patent_count
        Shunpei Yamazaki          6787
         Kia Silverbrook          4778
                 Tao Luo          4490
         Jonathan P. Ive          2947
                Junyi Li          2881
           Kangguo Cheng          2835
Frederick E. Shelton, IV          2723
              Peter Gaal          2554
      Duncan Robert Kerr          2534
        Bartley K. Andre          2478

--- Q2_Top_Companies ---
                                       name  patent_count
              Samsung Electronics Co., Ltd.        174536
International Business Machines Corporation        164083
                     Canon Kabushiki Kaisha         91331
                           Sony Corporation         62911
                            Fujitsu Limited         56343
                   Kabushiki Kaisha Toshiba         53607
                          Intel Corporation         53219
                   Gener